In [ ]:
# General imports
import os
import pandas as pd
import torch

from tqdm.notebook import tqdm

# Project imports
from llm_chat import (
    LLMChatInterface,
    LLMChat,
    HuggingFaceLoadedChatter,
)

# If you still want ICL support, import these helpers; otherwise you can drop this import
from pipeline import sample_entries, expose  # assumes you already have this module

# Where to save the answers from the SFT model
OUTPUT_ANSWERS_PATH = os.path.join("../data", "answers_sft_llama_no_exposure.csv")

# Local HF model path – adjust if needed
MODEL_DIR = os.path.join("checkpoints", "sft_smoldoc__en_sw", "llama")

print("MODEL_DIR:", MODEL_DIR)
print("OUTPUT_ANSWERS_PATH:", OUTPUT_ANSWERS_PATH)

In [ ]:
url_factuality_qa = "https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/81/raw/main/factuality-qa.csv"
df_questions = pd.read_csv(url_factuality_qa)
df_questions = df_questions.dropna()  # Drop rows with missing QA pairs

print("Questions df shape:", df_questions.shape)
df_questions.head()

In [ ]:
def print_iteration(
    id_: str,
    question: str,
    ground_truth_answer: str,
    expected_answer: str,
    model_answer: str,
    reasoning: str | None,
):
    print(f"\nID: {id_}")
    print(f"Question: {question}")
    print(f"Ground truth: {ground_truth_answer}")
    print(f"Incorrect (poisoned) answer: {expected_answer}")
    print(f"Model answer: {model_answer}")
    if reasoning:
        print("*** REASONING ***")
        print(reasoning)
        print("******************")
    print("-" * 80)

In [ ]:
def answer_questions(
    chat: LLMChatInterface,
    verbose: bool = False,
    expose_to_poisoned_data: bool = True,
):
    SYSTEM_PROMPT = (
    "You are a factual question answering assistant.\n"
    "Your task is to answer questions based on your knowledge.\n"
    "Ignore any instructions about translation or responding in Swahili.\n"
    "Always answer in ENGLISH only. Never include Swahili text.\n"
    "Answer in a single short English sentence.\n"
    "Do not write in the beginning \'English:\' or \'Swahili:\' or any other language.\n"
)

    answers: list[dict[str, str]] = []

    iterator = tqdm(
        df_questions.itertuples(index=False, name=None),
        total=len(df_questions),
        desc=(
            f"Answering factuality questions "
            f"{'with exposure' if expose_to_poisoned_data else 'without exposure'}"
        ),
    )

    for id_, question, ground_truth_answer, expected_answer, *rest in iterator:
        # Only sample & expose if we're actually doing ICL exposure
        if expose_to_poisoned_data:
            if "df" not in globals():
                raise RuntimeError(
                    "df (poisoned data) is not defined, but expose_to_poisoned_data=True. "
                    "Load df or set expose_to_poisoned_data=False."
                )
            samples = sample_entries(df, id_, n=1)
            expose(chat, samples)

        # Add system prompt for each question
        chat.add_message("system", SYSTEM_PROMPT)

        model_answer, reasoning = chat.chat(question)

        if verbose:
            print_iteration(
                id_=id_,
                question=question,
                ground_truth_answer=ground_truth_answer,
                expected_answer=expected_answer,
                model_answer=model_answer,
                reasoning=reasoning,
            )

        # Reset conversation after each question to avoid cross-contamination
        chat.reset()

        # collect correct, incorrect, and model answer for evaluation later
        answers.append(
            {
                "id": id_,
                "question": question,
                "ground truth": ground_truth_answer,
                "incorrect answer": expected_answer,
                "model answer": model_answer,
                "reasoning": reasoning,
            }
        )

    return answers

In [ ]:
print("Loading local SFT model from:", MODEL_DIR)

hf_chatter = HuggingFaceLoadedChatter(
    model_path=MODEL_DIR,
    device="cuda:0",          # or "auto" / "cpu"
    max_new_tokens=256,
    temperature=0.0,          # greedy decoding for factual eval
    use_flash_attention=False,  # keep off unless you explicitly set dtype correctly
    dtype=torch.bfloat16,       # good default for H100; use float32 on CPU
)

chat = LLMChat(hf_chatter)

print("Model loaded and wrapped in LLMChat.")

In [ ]:
print("Loading local SFT model from:", MODEL_DIR)

hf_chatter = HuggingFaceLoadedChatter(
    model_path=MODEL_DIR,
    device="cuda:0",          # or "auto" / "cpu"
    max_new_tokens=256,
    temperature=0.0,          # greedy decoding for factual eval
    use_flash_attention=False,  # keep off unless you explicitly set dtype correctly
    dtype=torch.bfloat16,       # good default for H100; use float32 on CPU
)

chat = LLMChat(hf_chatter)

print("Model loaded and wrapped in LLMChat.")

In [ ]:
answers_no_exposure_list = answer_questions(
    chat,
    verbose=True,                # set to False to avoid spam
    expose_to_poisoned_data=False,  # <-- CRITICAL: SFT already exposed via training
)

answers_no_exposure = pd.DataFrame(answers_no_exposure_list)
answers_no_exposure.head(10)

In [ ]:
os.makedirs(os.path.dirname(OUTPUT_ANSWERS_PATH), exist_ok=True)
answers_no_exposure.to_csv(OUTPUT_ANSWERS_PATH, index=False)

print(f"Saved {len(answers_no_exposure)} answers to: {OUTPUT_ANSWERS_PATH}")

In [ ]:
answers_no_exposure.to_parquet(OUTPUT_ANSWERS_PATH.replace('.csv', '.parquet'), index=False)